# Extra Trees: experiment

네 가지 실험 × 거시지표 포함·제외 8개 모델을 공통 기준 모델 2개와 비교합니다.

현재 수정치·달력 정렬 자료이며 과거 공개일 및 빈티지는 미검증입니다. origin 이후 정답은 학습에서 제외하지만 역사적 as-of 재현 백테스트를 보장하지 않습니다.

WAPE 단위는 %입니다. 검증 h=2~5의 전체 절대오차 합 / 실제금액 합으로 설정을 선정했습니다. 최종 테스트는 2025Q4 고정 모델의 h=1~3입니다. 결측 정답은 예측에 보존하고 평가에서만 제외합니다. 2027년 예측은 2025Q4 저장 모델의 연결 점검이며 운영용 재학습 결과가 아닙니다.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT = Path.cwd().resolve()
if not (ROOT / 'configs').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from oinq.modeling.supervised.extra_trees import run_training, LIMITATION
OUT = ROOT / '.private/extra_trees/extra_trees_20261109_02'
EXPERIMENT = 'experiment'
print('실행 결과:', OUT)
print(LIMITATION)


실행 결과: D:\hjs\five-stack\.private\extra_trees\extra_trees_20261109_02
현재 수정치·달력 정렬 자료이며 과거 공개일 및 빈티지는 미검증입니다. origin 이후 정답은 학습에서 제외하지만 역사적 as-of 재현 백테스트를 보장하지 않습니다.


## 공통 데이터 및 기간

manifest, features.yaml, splits.yaml의 입력 순서와 분할을 유지합니다. 각 origin마다 전처리를 새로 fit합니다. 최초 검증 origin=2020Q4이며 target 기준 분할이므로 2022년 이전 origin이 포함됩니다. 원본 결측·표본은 변경하지 않습니다.

In [2]:
audit = json.loads((OUT / 'data_audit.json').read_text(encoding='utf-8'))
print(pd.DataFrame({k:{x:v[x] for x in ['rows','columns','observed_targets','missing_targets','target_start','target_end','origin_start','origin_end','duplicate_keys']} for k,v in audit['partitions'].items()}).to_string())
print('입력 개수:', {k:len(v) for k,v in audit['feature_columns'].items()})
print('자료형 및 결측 수:')
for name, partition in audit['partitions'].items():
    print(name)
    print(pd.DataFrame({'dtype':partition['dtypes'], 'nulls':partition['null_counts']}).to_string())


                   train validation final_test
rows               28800      12800        480
columns               87         87         87
observed_targets   23290      11455        447
missing_targets     5510       1345         33
target_start      2013Q1     2022Q1     2026Q1
target_end        2021Q4     2025Q4     2026Q3
origin_start      2011Q4     2020Q4     2025Q4
origin_end        2021Q3     2025Q3     2025Q4
duplicate_keys         0          0          0
입력 개수: {'with_macro': 57, 'without_macro': 45}
자료형 및 결측 수:
train
                                                      dtype  nulls
country_code                                         string      0
region                                               string      0
hs_code                                              string      0
target_year                                           Int64      0
target_quarter_number                                 Int64      0
horizon_quarters                                      Int64    

## 후보값 및 실제 학습 호출

나머지 설정은 공통값으로 유지합니다. 가지치기 후보값은 최초 검증 origin의 학습 데이터에서만 생성하여 고정했습니다. 아래 RUN_TRAINING을 켜면 공통 모듈을 호출합니다. 기존 저장 경로가 있으면 덮어쓰기를 거부합니다. 재실행은 publish=False, notebooks=False로 별도 결과 폴더를 만듭니다.

In [3]:
definitions = json.loads((OUT / 'candidate_definitions.json').read_text(encoding='utf-8'))
print('alpha 생성 학습 종료:', definitions['alpha_fit_last_target'])
for mode, candidates in definitions['candidates'].items():
    print(mode)
    print(json.dumps({k:v for k,v in candidates.items() if EXPERIMENT=='experiment' or k=='baseline' or k.startswith(EXPERIMENT+'_')}, indent=2, ensure_ascii=False))
RUN_TRAINING = False
if RUN_TRAINING:
    selected_experiments = ('depth','leaf','features','pruned') if EXPERIMENT=='experiment' else (EXPERIMENT,)
    rerun = run_training(ROOT, experiments=selected_experiments, publish=False, forecast=False, notebooks=False)


alpha 생성 학습 종료: 2020Q4
with_macro
{
  "baseline": {
    "n_estimators": 16,
    "criterion": "squared_error",
    "max_depth": null,
    "min_samples_leaf": 1,
    "max_features": 1.0,
    "ccp_alpha": 0.0,
    "bootstrap": false,
    "random_state": 42,
    "n_jobs": 6
  },
  "depth_0": {
    "n_estimators": 16,
    "criterion": "squared_error",
    "max_depth": 8,
    "min_samples_leaf": 1,
    "max_features": 1.0,
    "ccp_alpha": 0.0,
    "bootstrap": false,
    "random_state": 42,
    "n_jobs": 6
  },
  "depth_1": {
    "n_estimators": 16,
    "criterion": "squared_error",
    "max_depth": 16,
    "min_samples_leaf": 1,
    "max_features": 1.0,
    "ccp_alpha": 0.0,
    "bootstrap": false,
    "random_state": 42,
    "n_jobs": 6
  },
  "depth_2": {
    "n_estimators": 16,
    "criterion": "squared_error",
    "max_depth": null,
    "min_samples_leaf": 1,
    "max_features": 1.0,
    "ccp_alpha": 0.0,
    "bootstrap": false,
    "random_state": 42,
    "n_jobs": 6
  },
  "leaf_0": 

## 이동 검증 및 선정 근거

거시지표 조건별 검증 WAPE가 가장 작은 후보를 선정했으며, 동률은 후보 ID 순입니다. 테스트 결과에 따른 설정 변경은 없습니다.

In [4]:
candidate_metrics = pd.read_csv(OUT / 'validation_candidate_metrics.csv')
mask = candidate_metrics.candidate.eq('baseline') | (candidate_metrics.candidate.str.startswith(EXPERIMENT+'_') if EXPERIMENT!='experiment' else True)
print(candidate_metrics.loc[mask].to_string(index=False))
lock = json.loads((OUT / 'selection_lock.json').read_text(encoding='utf-8'))
print(json.dumps([s for s in lock['selection'] if EXPERIMENT=='experiment' or s['model_id']=='extra_trees_'+EXPERIMENT], indent=2, ensure_ascii=False))


   macro_mode  candidate scope      WAPE          MAE         RMSE       R2  n_predictions  n_evaluated
   with_macro   baseline  h1_5 25.545532 2.753923e+06 1.723238e+07 0.900644          12800        11455
   with_macro   baseline  h2_5 27.043366 2.915396e+06 1.806895e+07 0.890763          10240         9164
   with_macro    depth_0  h1_5 25.342880 2.732076e+06 1.655748e+07 0.908274          12800        11455
   with_macro    depth_0  h2_5 26.759198 2.884761e+06 1.738136e+07 0.898918          10240         9164
   with_macro    depth_1  h1_5 24.839857 2.677848e+06 1.661629e+07 0.907621          12800        11455
   with_macro    depth_1  h2_5 26.395189 2.845520e+06 1.750580e+07 0.897466          10240         9164
   with_macro    depth_2  h1_5 25.545532 2.753923e+06 1.723238e+07 0.900644          12800        11455
   with_macro    depth_2  h2_5 27.043366 2.915396e+06 1.806895e+07 0.890763          10240         9164
   with_macro features_0  h1_5 25.420550 2.740449e+06 1.708900e+

## 최종 8개 모델 비교·평가 건수·WAPE 순위

공통 기준 Extra Trees는 별도 표에 표시하고 공유 8개 모델에서 제외합니다. 검증은 h=2~5, 테스트는 h=1~3이므로 두 지표의 예측 범위가 다릅니다.

In [5]:
summary = pd.read_csv(OUT / 'model_comparison.csv')
selected = summary if EXPERIMENT=='experiment' else summary[summary.model_id.eq('extra_trees_'+EXPERIMENT)]
print(selected.sort_values('validation_WAPE').to_string(index=False))
print('공통 기준 모델:')
print(pd.read_csv(OUT / 'baseline_comparison.csv').to_string(index=False))


            model_id    macro_mode                                                                                                                                                                                         hyperparameters dataset_version train_end  candidate  validation_n_predictions  validation_n_evaluated  validation_WAPE  validation_MAE  validation_RMSE  validation_R2  test_WAPE     test_MAE    test_RMSE  test_R2  n_predictions  n_evaluated                                         test_evaluation_key_hash  historical_asof_verified                                                                                     limitation                                                                 model_path  reload_exact  validation_WAPE_rank  test_WAPE_rank
    extra_trees_leaf without_macro                {"bootstrap": false, "ccp_alpha": 0.0, "criterion": "squared_error", "max_depth": null, "max_features": 1.0, "min_samples_leaf": 4, "n_estimators": 16, "n_jobs": 6, "random_sta

## 실제 가지치기 및 모델 복잡도

양의 alpha 후보와 가지치기 전 모델의 노드 수·깊이·리프 수를 비교합니다. 최적 alpha가 0인 경우도 성능 선택 결과 그대로 보존합니다.

In [6]:
complexity = pd.read_csv(OUT / 'validation_complexity.csv')
mask = complexity.candidate.eq('baseline') | (complexity.candidate.str.startswith(EXPERIMENT+'_') if EXPERIMENT!='experiment' else True)
print(complexity.loc[mask].groupby(['macro_mode','candidate'])[['mean_nodes','mean_depth','mean_leaves']].mean().to_string())
print(pd.read_csv(OUT / 'final_complexity.csv').to_string(index=False))


                           mean_nodes  mean_depth   mean_leaves
macro_mode    candidate                                        
with_macro    baseline    37364.97500   46.009375  18682.987500
              depth_0       310.25000    8.000000    155.625000
              depth_1      4226.36875   16.000000   2113.684375
              depth_2     37364.97500   46.009375  18682.987500
              features_0  43677.21875   46.415625  21839.109375
              features_1  39286.34375   46.109375  19643.671875
              features_2  37364.97500   46.009375  18682.987500
              leaf_0      37364.97500   46.009375  18682.987500
              leaf_1      10534.71875   41.731250   5267.859375
              leaf_2       2407.35625   34.650000   1204.178125
              pruned_0    37364.97500   46.009375  18682.987500
              pruned_1     7618.71875   33.246875   3809.859375
              pruned_2     1895.32500   24.071875    948.162500
              pruned_3      173.01250   

## 모델 저장·재로딩·2027년 연결 검증

공통 로더로 다시 읽어 전체 480행 예측의 완전 일치를 검증했습니다. 2027년은 기존 공통 예측 엔진으로 연결만 점검하며 2026Q3 운영용 재학습을 하지 않았습니다.

In [7]:
print(pd.read_csv(OUT / 'reload_verification.csv').to_string(index=False))
print(json.dumps(json.loads((OUT / 'forecast_verification.json').read_text(encoding='utf-8')), indent=2, ensure_ascii=False))
print((OUT / 'run_metadata.json').read_text(encoding='utf-8'))


            model_id    macro_mode  reload_exact  rows  max_absolute_difference                                                                 model_path
   extra_trees_depth    with_macro          True   480                      0.0       artifacts\shared_models\extra_trees_depth\with_macro\pipeline.joblib
    extra_trees_leaf    with_macro          True   480                      0.0        artifacts\shared_models\extra_trees_leaf\with_macro\pipeline.joblib
extra_trees_features    with_macro          True   480                      0.0    artifacts\shared_models\extra_trees_features\with_macro\pipeline.joblib
  extra_trees_pruned    with_macro          True   480                      0.0      artifacts\shared_models\extra_trees_pruned\with_macro\pipeline.joblib
   extra_trees_depth without_macro          True   480                      0.0    artifacts\shared_models\extra_trees_depth\without_macro\pipeline.joblib
    extra_trees_leaf without_macro          True   480                